In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("titanic.csv")
df.head()

,PassengerId,Name,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Survived
0,1,"Braund, Mr. Owen Harris",3,male,22.0,1,0,A/5 21171,7.2500,NaN,S,0
1,2,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,female,38.0,1,0,PC 17599,71.2833,C85,C,1
2,3,"Heikkinen, Miss. Laina",3,female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1
3,4,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,female,35.0,1,0,113803,53.1000,C123,S,1
4,5,"Allen, Mr. William Henry",3,male,35.0,0,0,373450,8.0500,NaN,S,0


In [3]:
df.drop(['PassengerId','Name','SibSp','Parch','Ticket','Cabin','Embarked'],axis='columns',inplace=True)
df.head()

,Pclass,Sex,Age,Fare,Survived
0,3,male,22.0,7.2500,0
1,1,female,38.0,71.2833,1
2,3,female,26.0,7.9250,1
3,1,female,35.0,53.1000,1
4,3,male,35.0,8.0500,0


In [4]:
inputs = df.drop('Survived',axis='columns')
target = df.Survived

In [5]:
dummies = pd.get_dummies(inputs.Sex)
dummies.head(3)

,female,male
0,False,True
1,True,False
2,True,False


In [6]:
inputs = pd.concat([inputs,dummies],axis='columns')
inputs.head(3)

,Pclass,Sex,Age,Fare,female,male
0,3,male,22.0,7.2500,False,True
1,1,female,38.0,71.2833,True,False
2,3,female,26.0,7.9250,True,False


I am dropping male column as well because of dummy variable trap theory. One column is enough to repressent male vs female

In [7]:
inputs.drop(['Sex','male'],axis='columns',inplace=True)
inputs.head(3)

,Pclass,Age,Fare,female
0,3,22.0,7.2500,False
1,1,38.0,71.2833,True
2,3,26.0,7.9250,True


In [8]:
inputs.columns[inputs.isna().any()]

Index(['Age'], dtype='object')

In [9]:
inputs.Age[:10]

0    22.0
1    38.0
2    26.0
3    35.0
4    35.0
5     NaN
6    54.0
7     2.0
8    27.0
9    14.0
Name: Age, dtype: float64

In [10]:
inputs.Age = inputs.Age.fillna(inputs.Age.mean())
inputs.head()

,Pclass,Age,Fare,female
0,3,22.0,7.2500,False
1,1,38.0,71.2833,True
2,3,26.0,7.9250,True
3,1,35.0,53.1000,True
4,3,35.0,8.0500,False


In [11]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(inputs,target,test_size=0.3)

In [12]:
len(X_train)

623

In [13]:
len(X_test)

268

In [14]:
len(inputs)

891

In [15]:
from sklearn.naive_bayes import GaussianNB
model = GaussianNB()

In [16]:
model.fit(X_train,y_train)

GaussianNB()

In [17]:
model.score(X_test,y_test)

0.7723880597014925

In [18]:
X_test[0:10]

,Pclass,Age,Fare,female
348,3,3.000000,15.9000,False
464,3,29.699118,8.0500,False
260,3,29.699118,7.7500,False
120,2,21.000000,73.5000,False
2,3,26.000000,7.9250,True
162,3,26.000000,7.7750,False
685,2,25.000000,41.5792,False
15,2,55.000000,16.0000,True
613,3,29.699118,7.7500,False
231,3,29.000000,7.7750,False


In [19]:
y_test[0:10]

348    1
464    0
260    0
120    0
2      1
162    0
685    0
15     1
613    0
231    0
Name: Survived, dtype: int64

In [20]:
model.predict(X_test[0:10])

array([0, 0, 0, 0, 1, 0, 0, 1, 0, 0], dtype=int64)

In [21]:
model.predict_proba(X_test[:10])

array([[0.94140313, 0.05859687],
       [0.97260141, 0.02739859],
       [0.97253242, 0.02746758],
       [0.76314712, 0.23685288],
       [0.46981569, 0.53018431],
       [0.97117236, 0.02882764],
       [0.92257762, 0.07742238],
       [0.26330241, 0.73669759],
       [0.97253242, 0.02746758],
       [0.97232822, 0.02767178]])

In [22]:
from sklearn.model_selection import cross_val_score
cross_val_score(GaussianNB(),X_train, y_train, cv=5)

array([0.824     , 0.704     , 0.776     , 0.79032258, 0.78225806])